In [ ]:
import os
import yaml
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()

from sae_tools.data_loader import get_adapter
from sae_tools.model import (
    load_sae_predictions_pt
)
from transformers import AutoTokenizer
from sae_tools.dashboard.viewer import FeatureActivationViewer

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['DejaVu Sans']


In [ ]:
BASE_DIR = Path.cwd()
MODEL_ROOT = os.getenv("MODEL_ROOT")
SAE_ROOT = os.getenv("SAE_ROOT")
DATASET_ROOT = os.getenv("DATASET_ROOT")

OUTPUT_DIR = BASE_DIR / "results/Guard_Qwen3-8B_20260206_1005"
PT_DIR = OUTPUT_DIR / "predictions"
CHARTS_DIR = os.path.join(OUTPUT_DIR, "charts")
SUMMARY_FILE = os.path.join(OUTPUT_DIR, "safe_feature_combination_summary.txt")

os.makedirs(CHARTS_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Output directory: {OUTPUT_DIR}")
print(f"Charts directory: {CHARTS_DIR}")
print(f"Output file: {SUMMARY_FILE}")
print("=" * 80)

In [ ]:
# prompt
# DATASET_NAME = "ToxicChat"
# DATASET_NAME = "OpenAIMod"
DATASET_NAME = "Aegis1.0"
# DATASET_NAME = "Aegis2.0"
# DATASET_NAME = "SimpleSafetyTest"
# DATASET_NAME = "HarmBench"
# DATASET_NAME = "WildGuardTest"

# response
# DATASET_NAME = "BeaverTails"
# DATASET_NAME = "BeaverTailsAugmented"
# DATASET_NAME = "Aegis2.0"
# DATASET_NAME = "SafeRLHF"
# DATASET_NAME = "WildGuardMix"

DATASETS_CONFIG = BASE_DIR / "configs/datasets/datasets_prompt.yaml"
with open(DATASETS_CONFIG, 'r', encoding='utf-8') as f:
    dataset_config = yaml.safe_load(f)
datasets = dataset_config.get('datasets', [])

DATASET_PATH = None
for dataset_info in datasets:
    dataset_name = dataset_info.get('name')
    if dataset_name == DATASET_NAME:
        DATASET_PATH = os.path.join(DATASET_ROOT, dataset_info.get('folder'))
        DATA_TYPE = dataset_info.get('type')
        LABEL_TYPE = f"{DATA_TYPE}_label"
        print(f"Dataset path: {DATASET_PATH}")
        print(f"Label type: {LABEL_TYPE}")
        break
if DATASET_PATH is None:
    raise ValueError(f"Dataset {DATASET_NAME} not found in {DATASETS_CONFIG}")

PT_FILE = PT_DIR / f"{DATASET_NAME}.pt"

In [ ]:
MODEL_NAME = "Qwen/Qwen3-8B"
model_path = os.path.join(MODEL_ROOT, MODEL_NAME)

# Load SAE Activations / Dataset

In [ ]:
sparse_data = load_sae_predictions_pt(PT_FILE)
num_samples = len(sparse_data['seq_lens'])

print(f"Loaded sparse activation data, number of samples: {num_samples}")
adapter = get_adapter(DATASET_NAME)
dataset = adapter.load(DATASET_PATH, num_samples)

metadata_list = [item for item in dataset]
print(f"Loaded {len(metadata_list)} metadata records")

In [ ]:
sparse_data

# Feature Activation Viewer

Use Feature Activation Viewer to view the activation positions and context of specified SAE features

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    model_path, 
    trust_remote_code=True
)

viewer = FeatureActivationViewer(
    sparse_data,
    metadata_list,
    tokenizer
)

print(f"Tokenizer loaded, model path: {model_path}")

In [ ]:
feature_id = 4744
max_display = 20

viewer.show_feature_activations(
    feature_id=feature_id,
    max_display=max_display,
    show_full_text=True,
    container_width="80%"
)

In [ ]:
markdown_text = viewer.get_markdown_text_with_activations(
    feature_id=feature_id,
    max_display=max_display
)
print(markdown_text)